In [ ]:
# 01 – Data preprocessing & differential expression (GSE266618)

#Paper:
#Datlinger P et al. *Systematic discovery of CRISPR-boosted CAR T cell immunotherapies.*
#Nature 2025;646(8086):963–972. DOI: 10.1038/s41586-025-9507-y

#Dataset:
#- Bulk RNA-seq counts: `GSE266618_counts.csv.gz` (Supplementary file)
#- GEO series matrix (for sample metadata): `GSE266618_series_matrix.txt` 


In [21]:
import os
import numpy as np
import pandas as pd

from scipy import stats
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)

DATA_RAW = "data/raw"
DATA_PROCESSED = "data/processed"
RESULTS = "results"

os.makedirs(DATA_PROCESSED, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)


In [22]:
import os

os.chdir("/Users/acastano/Desktop/crispr_carT_AI_analysis")
print("CWD now:", os.getcwd())
!ls


CWD now: /Users/acastano/Desktop/crispr_carT_AI_analysis
data              notebooks         results           zshrc_backup.txt
environment.yml   PROJECT_MEMORY.md scripts
figures           README.md         Untitled.ipynb


In [23]:
import pandas as pd

expr_counts = pd.read_csv(
    "data/raw/GSE266618_counts.csv",
    index_col=0
)

expr_counts.shape
expr_counts.head()


,CART0077_RNAseq_NgsRun1_001,CART0077_RNAseq_NgsRun1_002,CART0077_RNAseq_NgsRun1_003,CART0077_RNAseq_NgsRun1_004,CART0077_RNAseq_NgsRun1_005,CART0077_RNAseq_NgsRun1_006,CART0077_RNAseq_NgsRun1_007,CART0077_RNAseq_NgsRun1_008,CART0077_RNAseq_NgsRun1_009,CART0077_RNAseq_NgsRun1_010,...,CART0077_RNAseq_NgsRun1_051,CART0077_RNAseq_NgsRun1_052,CART0077_RNAseq_NgsRun1_053,CART0077_RNAseq_NgsRun1_054,CART0077_RNAseq_NgsRun1_055,CART0077_RNAseq_NgsRun1_056,CART0077_RNAseq_NgsRun1_057,CART0077_RNAseq_NgsRun1_058,CART0077_RNAseq_NgsRun1_059,CART0077_RNAseq_NgsRun1_060
gene,,,,,,,,,,,,,,,,,,,,,
ENSG00000223972,0,0,0,1,2,5,0,3,0,0,...,1,1,1,4,3,1,1,1,0,2
ENSG00000227232,189,257,124,184,189,320,136,157,78,97,...,123,145,192,224,181,159,136,115,246,217
ENSG00000278267,8,3,4,10,5,14,7,0,3,2,...,5,5,11,7,11,7,10,3,5,6
ENSG00000243485,0,0,0,0,0,0,0,0,0,5,...,1,0,0,2,0,3,0,0,0,0
ENSG00000284332,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
expr_counts.shape


(60675, 60)

In [25]:
expr_counts.index[:10]
expr_counts.columns[:10]


Index(['CART0077_RNAseq_NgsRun1_001', 'CART0077_RNAseq_NgsRun1_002',
       'CART0077_RNAseq_NgsRun1_003', 'CART0077_RNAseq_NgsRun1_004',
       'CART0077_RNAseq_NgsRun1_005', 'CART0077_RNAseq_NgsRun1_006',
       'CART0077_RNAseq_NgsRun1_007', 'CART0077_RNAseq_NgsRun1_008',
       'CART0077_RNAseq_NgsRun1_009', 'CART0077_RNAseq_NgsRun1_010'],
      dtype='object')

In [26]:
#Normalize Expression
lib_sizes = expr_counts.sum(axis=0)
cpm = expr_counts.divide(lib_sizes, axis=1) * 1e6
expr_logcpm = np.log2(cpm + 1)

expr_logcpm.shape
expr_logcpm.iloc[:5, :5]


,CART0077_RNAseq_NgsRun1_001,CART0077_RNAseq_NgsRun1_002,CART0077_RNAseq_NgsRun1_003,CART0077_RNAseq_NgsRun1_004,CART0077_RNAseq_NgsRun1_005
gene,,,,,
ENSG00000223972,0.000000,0.000000,0.000000,0.039740,0.104703
ENSG00000227232,2.851685,3.091911,2.190452,2.617975,3.020289
ENSG00000278267,0.337099,0.121486,0.157022,0.355340,0.248757
ENSG00000243485,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000284332,0.000000,0.000000,0.000000,0.000000,0.000000


In [27]:
#Parse metadata from the series matrix
series_path = "data/raw/GSE266618_series_matrix.txt"


In [28]:
#Extract the “title” and “geo_accession” fields:
sample_meta_raw = {}

with open(series_path, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("!Sample_"):
            parts = line.split("\t")
            key = parts[0].replace("!Sample_", "")  # e.g. "title", "geo_accession"
            values = parts[1:]
            sample_meta_raw[key] = values
        elif line.startswith("!series_matrix_table_begin"):
            break

metadata = pd.DataFrame(index=sample_meta_raw["geo_accession"])
metadata["title"] = sample_meta_raw["title"]
metadata.head()


,title
"""GSM8252546""","""CD4, 0h, SafeHarbor, CART0077_D1"""
"""GSM8252547""","""CD4, 0h, RHOG, CART0077_D1"""
"""GSM8252548""","""CD4, 0h, SafeHarbor, CART0077_D2"""
"""GSM8252549""","""CD4, 0h, RHOG, CART0077_D2"""
"""GSM8252550""","""CD4, 0h, SafeHarbor, CART0077_D3"""


In [29]:
#Extract biological fields from the title

def parse_title(title):
    parts = [p.strip() for p in title.split(",")]
    cell_type, time_str, guide, donor = parts
    hours = int(time_str.replace("h", "").strip())
    return pd.Series({
        "cell_type": cell_type,
        "hours": hours,
        "guide": guide,
        "donor": donor
    })

parsed = metadata["title"].apply(parse_title)
metadata = pd.concat([metadata, parsed], axis=1)
metadata


,title,cell_type,hours,guide,donor
"""GSM8252546""","""CD4, 0h, SafeHarbor, CART0077_D1""","""CD4",0,SafeHarbor,"CART0077_D1"""
"""GSM8252547""","""CD4, 0h, RHOG, CART0077_D1""","""CD4",0,RHOG,"CART0077_D1"""
"""GSM8252548""","""CD4, 0h, SafeHarbor, CART0077_D2""","""CD4",0,SafeHarbor,"CART0077_D2"""
"""GSM8252549""","""CD4, 0h, RHOG, CART0077_D2""","""CD4",0,RHOG,"CART0077_D2"""
"""GSM8252550""","""CD4, 0h, SafeHarbor, CART0077_D3""","""CD4",0,SafeHarbor,"CART0077_D3"""
...,...,...,...,...,...
"""GSM8252601""","""CD8, 240h, RHOG, CART0077_D1""","""CD8",240,RHOG,"CART0077_D1"""
"""GSM8252602""","""CD8, 240h, SafeHarbor, CART0077_D2""","""CD8",240,SafeHarbor,"CART0077_D2"""
"""GSM8252603""","""CD8, 240h, RHOG, CART0077_D2""","""CD8",240,RHOG,"CART0077_D2"""
"""GSM8252604""","""CD8, 240h, SafeHarbor, CART0077_D3""","""CD8",240,SafeHarbor,"CART0077_D3"""


In [30]:
import os
os.getcwd()



'/Users/acastano/Desktop/crispr_carT_AI_analysis'

In [31]:
os.chdir("/Users/acastano/Desktop/crispr_carT_AI_analysis")
os.getcwd()


'/Users/acastano/Desktop/crispr_carT_AI_analysis'

In [32]:
!ls data/raw


GPL570-55999.txt            GSE266618_series_matrix.txt
GSE266618_counts.csv


In [33]:
import pandas as pd

series_path = "data/raw/GSE266618_series_matrix.txt"

sample_meta_raw = {}

with open(series_path, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("!Sample_"):
            parts = line.split("\t")
            key = parts[0].replace("!Sample_", "")
            values = parts[1:]
            sample_meta_raw[key] = values
        elif line.startswith("!series_matrix_table_begin"):
            break

print("Fields available:", list(sample_meta_raw.keys()))


Fields available: ['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'characteristics_ch1', 'treatment_protocol_ch1', 'growth_protocol_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'taxid_ch1', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_institute', 'contact_address', 'contact_city', 'contact_zip/postal_code', 'contact_country', 'data_row_count', 'instrument_model', 'library_selection', 'library_source', 'library_strategy', 'supplementary_file_1']


In [34]:
list(sample_meta_raw.keys())
sample_meta_raw["title"][:5]


['"CD4, 0h, SafeHarbor, CART0077_D1"',
 '"CD4, 0h, RHOG, CART0077_D1"',
 '"CD4, 0h, SafeHarbor, CART0077_D2"',
 '"CD4, 0h, RHOG, CART0077_D2"',
 '"CD4, 0h, SafeHarbor, CART0077_D3"']

In [35]:
#Rebuild sample_meta_raw
series_path = "data/raw/GSE266618_series_matrix.txt"

sample_meta_raw = {}
with open(series_path, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("!Sample_"):
            parts = line.split("\t")
            key = parts[0].replace("!Sample_", "")
            values = parts[1:]
            sample_meta_raw[key] = values
        elif line.startswith("!series_matrix_table_begin"):
            break

list(sample_meta_raw.keys())


['title',
 'geo_accession',
 'status',
 'submission_date',
 'last_update_date',
 'type',
 'channel_count',
 'source_name_ch1',
 'organism_ch1',
 'characteristics_ch1',
 'treatment_protocol_ch1',
 'growth_protocol_ch1',
 'molecule_ch1',
 'extract_protocol_ch1',
 'taxid_ch1',
 'description',
 'data_processing',
 'platform_id',
 'contact_name',
 'contact_email',
 'contact_institute',
 'contact_address',
 'contact_city',
 'contact_zip/postal_code',
 'contact_country',
 'data_row_count',
 'instrument_model',
 'library_selection',
 'library_source',
 'library_strategy',
 'supplementary_file_1']

In [36]:
import os
os.chdir("/Users/acastano/Desktop/crispr_carT_AI_analysis")
print(os.getcwd())
!ls data/raw


/Users/acastano/Desktop/crispr_carT_AI_analysis
GPL570-55999.txt            GSE266618_series_matrix.txt
GSE266618_counts.csv


In [37]:
import pandas as pd

expr_counts = pd.read_csv(
    "data/raw/GSE266618_counts.csv",
    index_col=0
)

expr_counts.shape, expr_counts.head()


((60675, 60),
                  CART0077_RNAseq_NgsRun1_001  CART0077_RNAseq_NgsRun1_002  \
 gene                                                                        
 ENSG00000223972                            0                            0   
 ENSG00000227232                          189                          257   
 ENSG00000278267                            8                            3   
 ENSG00000243485                            0                            0   
 ENSG00000284332                            0                            0   
 
                  CART0077_RNAseq_NgsRun1_003  CART0077_RNAseq_NgsRun1_004  \
 gene                                                                        
 ENSG00000223972                            0                            1   
 ENSG00000227232                          124                          184   
 ENSG00000278267                            4                           10   
 ENSG00000243485                            0   

In [38]:
sample_meta_raw = {}

with open("data/raw/GSE266618_series_matrix.txt", "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("!Sample_"):
            parts = line.split("\t")
            key = parts[0].replace("!Sample_", "")
            values = parts[1:]
            sample_meta_raw[key] = values
        elif line.startswith("!series_matrix_table_begin"):
            break

sample_meta_raw.keys()



dict_keys(['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'characteristics_ch1', 'treatment_protocol_ch1', 'growth_protocol_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'taxid_ch1', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_institute', 'contact_address', 'contact_city', 'contact_zip/postal_code', 'contact_country', 'data_row_count', 'instrument_model', 'library_selection', 'library_source', 'library_strategy', 'supplementary_file_1'])

In [39]:
n_samples_counts = expr_counts.shape[1]
titles_raw = pd.Series(sample_meta_raw["title"], name="title")
gsm_ids    = pd.Series(sample_meta_raw["geo_accession"], name="GSM")

print("Counts:", n_samples_counts, "Titles:", len(titles_raw), "GSMs:", len(gsm_ids))


Counts: 60 Titles: 60 GSMs: 60


In [40]:
metadata = pd.DataFrame({
    "sample_id": expr_counts.columns,
    "GSM": gsm_ids.values,
    "title": titles_raw.values
}).set_index("sample_id")

metadata.head()


,GSM,title
sample_id,,
CART0077_RNAseq_NgsRun1_001,"""GSM8252546""","""CD4, 0h, SafeHarbor, CART0077_D1"""
CART0077_RNAseq_NgsRun1_002,"""GSM8252547""","""CD4, 0h, RHOG, CART0077_D1"""
CART0077_RNAseq_NgsRun1_003,"""GSM8252548""","""CD4, 0h, SafeHarbor, CART0077_D2"""
CART0077_RNAseq_NgsRun1_004,"""GSM8252549""","""CD4, 0h, RHOG, CART0077_D2"""
CART0077_RNAseq_NgsRun1_005,"""GSM8252550""","""CD4, 0h, SafeHarbor, CART0077_D3"""


In [41]:
def parse_title(title):
    # Remove any surrounding quotes
    title = title.strip('"')
    parts = [p.strip() for p in title.split(",")]
    
    # Expect: cell_type, time, guide, donor
    cell_type, time_str, guide, donor = parts
    hours = int(time_str.replace("h", "").strip())
    
    return pd.Series({
        "cell_type": cell_type,
        "hours": hours,
        "guide": guide,
        "donor": donor
    })

parsed = metadata["title"].apply(parse_title)
metadata = pd.concat([metadata, parsed], axis=1)

metadata.head()


,GSM,title,cell_type,hours,guide,donor
sample_id,,,,,,
CART0077_RNAseq_NgsRun1_001,"""GSM8252546""","""CD4, 0h, SafeHarbor, CART0077_D1""",CD4,0,SafeHarbor,CART0077_D1
CART0077_RNAseq_NgsRun1_002,"""GSM8252547""","""CD4, 0h, RHOG, CART0077_D1""",CD4,0,RHOG,CART0077_D1
CART0077_RNAseq_NgsRun1_003,"""GSM8252548""","""CD4, 0h, SafeHarbor, CART0077_D2""",CD4,0,SafeHarbor,CART0077_D2
CART0077_RNAseq_NgsRun1_004,"""GSM8252549""","""CD4, 0h, RHOG, CART0077_D2""",CD4,0,RHOG,CART0077_D2
CART0077_RNAseq_NgsRun1_005,"""GSM8252550""","""CD4, 0h, SafeHarbor, CART0077_D3""",CD4,0,SafeHarbor,CART0077_D3


In [42]:
metadata["hours"].value_counts().sort_index()


hours
0      12
24     12
72     12
168    12
240    12
Name: count, dtype: int64

In [43]:
metadata["guide"].value_counts()


guide
SafeHarbor    30
RHOG          30
Name: count, dtype: int64

In [44]:
metadata["donor"].value_counts()


donor
CART0077_D1    20
CART0077_D2    20
CART0077_D3    20
Name: count, dtype: int64

In [45]:
import numpy as np

lib_sizes = expr_counts.sum(axis=0)
cpm = expr_counts.divide(lib_sizes, axis=1) * 1e6
expr_logcpm = np.log2(cpm + 1)

expr_logcpm.shape


(60675, 60)

In [46]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

def differential_expression(expr_df, metadata, mask_group1, mask_group2,
                            group1_name="group1", group2_name="group2"):
    samples1 = metadata.index[mask_group1]
    samples2 = metadata.index[mask_group2]
    
    X1 = expr_df.loc[:, samples1]
    X2 = expr_df.loc[:, samples2]
    
    mean1 = X1.mean(axis=1)
    mean2 = X2.mean(axis=1)
    log2_fc = mean2 - mean1
    
    # Welch’s t-test
    t_stats, pvals = stats.ttest_ind(X2.T, X1.T, equal_var=False, nan_policy="omit")
    
    # FDR correction
    _, pval_adj, _, _ = multipletests(pvals, method="fdr_bh")
    
    return pd.DataFrame({
        f"mean_{group1_name}": mean1,
        f"mean_{group2_name}": mean2,
        "log2_fc": log2_fc,
        "pval": pvals,
        "pval_adj": pval_adj
    })


In [47]:
early_mask = metadata["hours"] == 0
late_mask  = metadata["hours"] >= 168

de_late_vs_early = differential_expression(
    expr_logcpm, metadata,
    mask_group1=early_mask,
    mask_group2=late_mask,
    group1_name="early_0h",
    group2_name="late_168hplus"
)

de_late_vs_early.head()


,mean_early_0h,mean_late_168hplus,log2_fc,pval,pval_adj
gene,,,,,
ENSG00000223972,0.055567,0.049877,-0.005689,0.811482,NaN
ENSG00000227232,2.649494,2.853152,0.203658,0.187170,NaN
ENSG00000278267,0.242513,0.366924,0.124411,0.025860,NaN
ENSG00000243485,0.018491,0.013329,-0.005163,0.801665,NaN
ENSG00000284332,0.000000,0.000000,0.000000,NaN,NaN
